# 📘 Lesson 5 — Cross‑Validation
> #### 📚 Curso Intermediate Machine Learning

- Organizando pré‑processamento e modelagem em um único fluxo
- Hybrid Learning Notebook — Study + Reference + Hands-on

<br>



> #### 🎯 Objetivos

- Nesta lesson, você aprenderá:
    - Por que um validation set único pode gerar avaliações instáveis
    - Como Cross‑Validation reduz o impacto do acaso
    - Como usar `cross_val_score()` corretamente
    - Como interpretar scores por fold
    - Como integrar Cross‑Validation com Pipelines
    - Como comparar modelos usando a média dos folds

<br>

> Este notebook segue **exatamente o conteúdo da lesson do Kaggle**, com explicações adicionais para facilitar o aprendizado.

<br>

---

### 🟦 Glossário Técnico

- Consulta Rápida
    - **Cross‑Validation (CV)** - Técnica de avaliação que divide o dataset em múltiplos folds e repete o experimento várias vezes.
    - **Fold** - Subconjunto do dataset usado como validação em uma iteração da CV.
    - **K‑Fold** - Divisão do dataset em K partes iguais.
    - **cross_val_score()** - Função do scikit‑learn que executa Cross‑Validation automaticamente.
    - **MAE (Mean Absolute Error)** - Métrica usada no curso; quanto menor, melhor.
    - **Pipeline** - Fluxo integrado de pré‑processamento + modelo.
    - **neg_mean_absolute_error** - Forma como o scikit‑learn representa MAE (negativo), pois métricas devem ser maximizadas.

<br>

---


### 🟩 Mini‑Referência (API Style)

- Comandos Essenciais
    - **from sklearn.model_selection import cross_val_score** -> Executa Cross‑Validation.
    - **cross_val_score(model, X, y, cv=5, scoring="neg_mean_absolute_error")** -> Retorna um array com o MAE (negativo) de cada fold.
    - **Pipeline(steps=[("preprocessor", ...), ("model", ...)])** -> Encapsula pré‑processamento e modelo.
    - **SimpleImputer()** -> Preenche valores ausentes.
    - **RandomForestRegressor(n_estimators=50, random_state=0)** -> Modelo base usado na lesson.
    - **scores.mean()** -> Média dos scores dos folds.

<br>

---


### 🟦 Introdução — por que validação simples não é suficiente?

Até aqui, avaliamos modelos com um `**validation set**` (holdout). <br>
* Isso funciona, mas tem um problema: o score pode variar bastante dependendo de *quais linhas* caíram na validação.

<br>

> **Cross‑Validation** resolve isso repetindo o mesmo experimento várias vezes, com diferentes “pedaços” (folds) do dataset como validação.

- Objetivo desta lesson:
    - Entender **POR QUE** Cross‑Validation é necessária
    - Ver **COMO** ela melhora a avaliação do modelo
    - Usar Cross‑Validation **junto com Pipeline** (como o Kaggle faz)


<br>

---

### 🟩 Contexto do problema — ligação com Pipelines

#### Contexto

A partir da lesson anterior, o pré‑processamento fica encapsulado em um **Pipeline**.
- Isso é importante porque Cross‑Validation precisa repetir o fluxo completo (preprocess + model) em cada fold.

<br>

>Sem Pipeline, seria fácil cometer erros (ex.: imputar de um jeito no treino e de outro na validação).
Com Pipeline, o scikit‑learn garante consistência.

<br>

#### Explicação conceitual (book-style)

- **Holdout (validation set único):**
    - Treina em ~80% e valida em ~20%
    - Rápido
    - Pode ter “sorte/azar” dependendo do split

- **Cross‑Validation (K‑Fold):**
    - Divide o dataset em K folds
    - Repete K vezes: treina em K‑1 folds e valida no fold restante
    - Produz K scores → usamos a **média** como medida mais estável

- Trade‑off:
    - Mais confiável
    - Mais lento (treina K modelos)

<br>

---

### 🟨 Implementação passo a passo (igual ao Kaggle)

#### 1 - Importar bibliotecas + carregar dados

In [2]:
import sys
from pathlib import Path

notebook_path = Path().resolve()
for parent in notebook_path.parents:
    if (parent / "config.py").exists():
        sys.path.append(str(parent))
        break

import pandas as pd
from config import get_data_path

DATA_PATH = get_data_path()


In [3]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [4]:
train_data = pd.read_csv(DATA_PATH + "train.csv", index_col="Id")
test_data = pd.read_csv(DATA_PATH + "test.csv", index_col="Id")

train_data.dropna(axis=0, subset=["SalePrice"], inplace=True)
y = train_data.SalePrice
train_data.drop(["SalePrice"], axis=1, inplace=True)

numeric_cols = [
    cname for cname in train_data.columns
    if train_data[cname].dtype in ["int64", "float64"]
]

X = train_data[numeric_cols].copy()
X_test = test_data[numeric_cols].copy()

#### 2 Definir o Pipeline

In [5]:
my_pipeline = Pipeline(steps=[
    ("preprocessor", SimpleImputer()),
    ("model", RandomForestRegressor(n_estimators=50, random_state=0))
])

#### 3 Avaliação SEM Cross‑Validation (baseline)

In [6]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, train_size=0.8, random_state=0
)

my_pipeline.fit(X_train, y_train)
preds = my_pipeline.predict(X_valid)

mae_holdout = mean_absolute_error(y_valid, preds)
print("MAE (holdout validation set):", mae_holdout)

MAE (holdout validation set): 17999.59881278539


#### 4 Avaliação COM Cross‑Validation

In [7]:
scores = -1 * cross_val_score(
    my_pipeline,
    X,
    y,
    cv=5,
    scoring="neg_mean_absolute_error"
)

print("MAE scores (por fold):\n", scores)
print("\nAverage MAE score (across experiments):")
print(scores.mean())

MAE scores (por fold):
 [18549.88568493 17896.67034247 18462.68472603 16587.02472603
 19885.78630137]

Average MAE score (across experiments):
18276.410356164386


### 🟪 Observações pedagógicas do Copilot (o que o Kaggle assume)

1) **Por que Pipeline aqui é essencial?**
    - Em cada fold, o scikit-learn precisa:
    - ajustar o imputador só com o treino daquele fold
    - aplicar no holdout daquele fold
    - treinar o modelo
    - Pipeline garante que isso aconteça corretamente e sem vazamento.

<br>

2) **Como interpretar os scores por fold?**
    - Se os valores forem próximos, o modelo é mais “estável”.
    - Se varia muito, o desempenho depende bastante de quais linhas caem no holdout.

<br>

3) **Quando vale a pena usar Cross-Validation?**
    - Quando o dataset é menor ou quando você está tomando muitas decisões (features, parâmetros, modelos).
    - Se o treino é muito pesado, pode ser melhor usar holdout por velocidade.

<br>

4) **O que NÃO estamos fazendo aqui (de propósito):**
    - Não estamos tunando hiperparâmetros (isso é o exercise e capítulos seguintes).
    - Não estamos mudando o modelo.
    - Não estamos adicionando novas técnicas.


### 🟧 Conclusão — aprendizados da lesson

- ##### Conclusão
    - Holdout é rápido, mas pode ser “ruidoso” (depende do split).
    - Cross-Validation fornece uma medida mais confiável do desempenho.
    - Pipeline torna Cross-Validation simples e segura.
    - A média dos folds é a melhor forma de comparar alternativas.

<br>

Próximo passo (exercise do Kaggle):
- usar Cross-Validation para escolher um bom valor de `n_estimators`. -->